# SAM-WM frozen Kaggle benchmark

Run top-to-bottom. Exactly one model is trained: **SAM-WM**.

The execution order is Freiburg train/validation → validation-only deployment-seed preselection →
cryptographic freeze → Freiburg held-out once → Novi Sad zero-shot → FAIRUrbTemp zero-shot →
aggregate evidence → finalize the already-preselected SAM-WM deployment bundle.

Do not change architecture, objective, preprocessing, QC, graph construction, split, or
hyperparameters after the freeze.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.error
import urllib.request
from pathlib import Path

REPOSITORY = "AnnyaB/SAM-WM"
WORK_ROOT = Path("/kaggle/working")
REPO = WORK_ROOT / "SAM-WM"
SEEDS = (17, 29, 42, 73, 101)

def github_token() -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        value = None
    return value.strip() if isinstance(value, str) and value.strip() else None

def github_json(url: str, token: str | None) -> dict:
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "sam-wm-kaggle",
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"
    request = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.load(response)
    except urllib.error.HTTPError as exc:
        if exc.code in {401, 403, 404}:
            raise RuntimeError(
                "GitHub source is not accessible. For the private repository, add a Kaggle "
                "Secret named GITHUB_TOKEN with read access to AnnyaB/SAM-WM."
            ) from exc
        raise

def download_archive(url: str, dst: Path, token: str | None) -> None:
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "sam-wm-kaggle",
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"
    request = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(request, timeout=120) as response, dst.open("wb") as handle:
        shutil.copyfileobj(response, handle)

def safe_extract(archive: Path, dst: Path) -> Path:
    dst.mkdir(parents=True, exist_ok=True)
    root = dst.resolve()
    with tarfile.open(archive, mode="r:gz") as tf:
        members = tf.getmembers()
        for member in members:
            target = (root / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f"unsafe archive path: {member.name}")
            if member.issym() or member.islnk():
                raise RuntimeError(f"archive links are not accepted: {member.name}")
        tf.extractall(root)
    directories = [path for path in root.iterdir() if path.is_dir()]
    if len(directories) != 1:
        raise RuntimeError(f"unexpected GitHub archive layout: {directories}")
    return directories[0]

token = github_token()
meta = github_json(f"https://api.github.com/repos/{REPOSITORY}/commits/main", token)
SOURCE_SHA = str(meta["sha"])
if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_SHA):
    raise RuntimeError("GitHub returned an invalid commit SHA")

if REPO.exists():
    shutil.rmtree(REPO)

with tempfile.TemporaryDirectory(prefix="samwm-source-") as tmp_name:
    tmp = Path(tmp_name)
    archive = tmp / "source.tar.gz"
    download_archive(
        f"https://api.github.com/repos/{REPOSITORY}/tarball/{SOURCE_SHA}",
        archive,
        token,
    )
    extracted = safe_extract(archive, tmp / "extract")
    shutil.copytree(extracted, REPO)

os.chdir(REPO)
(REPO / "artifacts").mkdir(exist_ok=True)
(REPO / "artifacts" / "FROZEN_SOURCE_SHA.txt").write_text(
    SOURCE_SHA + "\n", encoding="utf-8"
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"],
    check=True,
)
subprocess.run(["make", f"PYTHON={sys.executable}", "verify"], check=True)
print("Frozen source SHA:", SOURCE_SHA)


In [ ]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before training.")
print("GPU:", torch.cuda.get_device_name(0))


## Train full SAM-WM — Freiburg train/validation only

This cell trains the same full SAM-WM architecture across the five frozen seeds.
It does not train baselines, alternative model families, or OOD-adapted variants.


In [ ]:
def run_python(*args: str) -> None:
    subprocess.run([sys.executable, *args], check=True)

run_python("research.py", "--out", "artifacts/research")

manifest = Path("artifacts/research/PRE_FREEZE_MANIFEST.json")
payload = json.loads(manifest.read_text(encoding="utf-8"))
assert payload["protocol"] == "SAM_WM_PRE_FREEZE_V2"
assert payload["model"] == "SAM-WM"
assert payload["heldout_or_ood_accessed"] is False

validation_rows = []
for seed in SEEDS:
    path = Path(f"artifacts/research/seed_{seed}/validation_metrics.json")
    row = json.loads(path.read_text(encoding="utf-8"))
    validation_rows.append(
        {
            "seed": seed,
            "mae": row["validation"]["mae"],
            "rmse": row["validation"]["rmse"],
            "bias": row["validation"]["bias"],
        }
    )
validation_rows


## Preselect deployment seed from Freiburg validation only

This selection happens before final/OOD access. It cannot use Freiburg held-out,
Novi Sad, or FAIRUrbTemp results.


In [ ]:
run_python("promote.py", "preselect")
selection_path = Path("artifacts/DEPLOYMENT_SELECTION.json")
selection = json.loads(selection_path.read_text(encoding="utf-8"))
assert selection["model"] == "SAM-WM"
assert selection["heldout_or_ood_used_for_selection"] is False
selection


## Freeze the reported SAM-WM run

Run this only when no more development changes will be made.


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

pre_freeze = REPO / "artifacts/research/PRE_FREEZE_MANIFEST.json"
selection_path = REPO / "artifacts/DEPLOYMENT_SELECTION.json"
checkpoints = {
    f"seed_{seed}": REPO / f"artifacts/research/seed_{seed}/best.pt"
    for seed in SEEDS
}
missing = [str(path) for path in checkpoints.values() if not path.exists()]
if missing:
    raise RuntimeError(f"SAM-WM checkpoints are incomplete: {missing}")

selection = json.loads(selection_path.read_text(encoding="utf-8"))
if selection["source_sha"] != SOURCE_SHA:
    raise RuntimeError("deployment selection source SHA does not match notebook source SHA")
if selection["pre_freeze_manifest_sha256"] != sha256_file(pre_freeze):
    raise RuntimeError("pre-freeze manifest changed after deployment preselection")

freeze = {
    "protocol": "SAM_WM_FINAL_FREEZE_V1",
    "model": "SAM-WM",
    "source_sha": SOURCE_SHA,
    "config_sha256": sha256_file(REPO / "config/train.yaml"),
    "pre_freeze_manifest_sha256": sha256_file(pre_freeze),
    "deployment_selection_sha256": sha256_file(selection_path),
    "seeds": list(SEEDS),
    "full_checkpoints": {
        name: sha256_file(path) for name, path in checkpoints.items()
    },
    "rule": (
        "No architecture, objective, preprocessing, QC, split, graph or hyperparameter "
        "change is allowed after this manifest for the reported held-out/OOD run."
    ),
}
freeze_path = REPO / "artifacts/FREEZE_MANIFEST.json"
freeze_path.write_text(
    json.dumps(freeze, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(freeze_path.read_text(encoding="utf-8"))


## Freiburg final test — open once per frozen SAM-WM seed


In [ ]:
for seed in SEEDS:
    run_python(
        "eval.py",
        "--checkpoint", f"artifacts/research/seed_{seed}/best.pt",
        "--data", "freiburg",
        "--split", "heldout",
        "--open-heldout",
        "--out", f"artifacts/eval/seed_{seed}",
    )


## OOD-1 — Novi Sad zero-shot


In [ ]:
for seed in SEEDS:
    run_python(
        "eval.py",
        "--checkpoint", f"artifacts/research/seed_{seed}/best.pt",
        "--data", "novisad",
        "--split", "heldout",
        "--open-heldout",
        "--out", f"artifacts/eval/seed_{seed}",
    )


## OOD-2 — FAIRUrbTemp unseen city zero-shot

Before running this cell, attach the extracted official DOI `10.48620/93247` files as a
Kaggle Input. Set `FAIR_ROOT` to that extracted directory and set `FAIR_CITY` using
metadata/coverage criteria only, before viewing any SAM-WM FAIRUrbTemp metric.


In [ ]:
FAIR_ROOT = os.environ.get("FAIR_ROOT", "").strip()
FAIR_CITY = os.environ.get("FAIR_CITY", "").strip()
if not FAIR_ROOT:
    raise RuntimeError("Set FAIR_ROOT to the extracted official FAIRUrbTemp DOI directory.")
if not FAIR_CITY:
    raise RuntimeError(
        "Set FAIR_CITY from preregistered metadata/coverage criteria before OOD scoring."
    )

for seed in SEEDS:
    run_python(
        "eval.py",
        "--checkpoint", f"artifacts/research/seed_{seed}/best.pt",
        "--data", "fairurbtemp",
        "--root", FAIR_ROOT,
        "--city", FAIR_CITY,
        "--split", "heldout",
        "--open-heldout",
        "--out", f"artifacts/eval/seed_{seed}",
    )


## Aggregate evidence and finalize the already-preselected SAM-WM checkpoint


In [ ]:
run_python("summarize.py", "--root", "artifacts/eval", "--out", "artifacts/summary.json")
run_python("promote.py", "finalize")

selected_seed = int(selection["selected_seed"])
plot_inputs = [
    f"artifacts/eval/seed_{selected_seed}/freiburg_heldout_metrics.json",
    f"artifacts/eval/seed_{selected_seed}/novisad_heldout_metrics.json",
    f"artifacts/eval/seed_{selected_seed}/fairurbtemp_heldout_metrics.json",
]
run_python("plot.py", *plot_inputs, "--out", "artifacts/figures")

print("Selected SAM-WM deployment seed:", selected_seed)
print(Path("artifacts/deployment/PROMOTION_MANIFEST.json").read_text(encoding="utf-8"))


## Stop here before deployment

Persist the entire `artifacts/` directory as Kaggle output. The next phase is real FortyGuard
provider replay, trained-checkpoint UI integration, public deployment, visual verification, and
the ≤3-minute working demo video. Do not change the frozen model after seeing final/OOD results.
